In [ ]:
# Lab 10, notebook 3: is the pipeline healthy today? Five checks on the newest day, each with a limit, an owner
# and a first action. Attached lakehouse: freshcart_lake. It runs last in the pipeline, after the labels.

### The checks, their limits, and who acts on each

This cell lists the five checks. Each has a limit, an owner (the person who acts when it alerts) and a first action. It also finds the newest day in tickets_raw, the table the copy job fills: every check below looks at that day. If a table does not exist yet, a check reports it instead of crashing.

In [ ]:
import json, statistics, time
from pyspark.sql import functions as F

CHECKS = {   # name: (limit, owner, first action)
    "pipeline is up to date": ("newest raw ticket is in clean or quarantine", "data engineer on call",
                               "Run the pipeline; if it fails, read the failed step's error"),
    "volume is normal": ("50% to 200% of the usual day", "data engineer on call",
                         "Compare with the source table; a half day usually means the copy stopped early"),
    "quarantine rate": ("at most 5% of the day's tickets", "data owner (support operations)",
                        "Read tickets_quarantine for that day: which rule, which channel"),
    "ticket mix": ("no AI category moves more than 10 points", "support operations lead",
                   "Check whether something real happened that day before blaming the model"),
    "labelling cost": ("latest run at most 5 times the usual run", "ML engineer",
                       "Check enrich_log: prompt_version and model_calls; was the prompt changed?"),
}
results = []

def record(name, value, ok):
    limit, owner, action = CHECKS[name]
    results.append({"check": name, "value": str(value), "limit": limit, "status": "OK" if ok else "ALERT",
                    "owner": owner, "first_action": action})

def table(name):
    """A table if it exists yet, else None: a check must report a missing table, never crash on it."""
    return spark.read.table(name) if spark.catalog.tableExists(name) else None

raw = table("tickets_raw")
day = raw.agg(F.max(F.to_date("created_at"))).first()[0] if raw is not None else None   # the newest day copied
print("newest day:", day)

### 1. Is the pipeline up to date?

It compares the newest ticket the copy job brought in with the newest ticket the quality check has handled (in tickets_clean or tickets_quarantine). If the handled one is older, a day was copied but not processed.

In [ ]:
parts = [t.select("created_at") for t in (table("tickets_clean"), table("tickets_quarantine")) if t is not None]
newest_raw = raw.agg(F.max("created_at")).first()[0] if raw is not None else None
newest_handled = parts[0].unionByName(parts[1]).agg(F.max("created_at")).first()[0] if len(parts) == 2 else None
record("pipeline is up to date", f"raw {newest_raw}, handled {newest_handled}",
       newest_raw is not None and newest_handled is not None and newest_handled >= newest_raw)

### 2. Is the volume normal?

It counts the newest day's tickets and compares them with the median day of the 30 days before (the middle value when the days are sorted). Between 50% and 200% of that is normal.

In [ ]:
daily = raw.groupBy(F.to_date("created_at").alias("d")).count() if raw is not None else None
today_row = daily.filter(F.col("d") == day).first() if daily is not None else None
today = today_row["count"] if today_row else 0
earlier_days = [r["count"] for r in daily.filter((F.col("d") < day) & (F.col("d") >= F.date_sub(F.lit(day), 30))).collect()] if daily is not None else []
if not earlier_days:                               # nothing to compare with yet
    record("volume is normal", f"{today} tickets (no earlier days to compare)", False)
else:
    usual = statistics.median(earlier_days)
    record("volume is normal", f"{today} tickets (usual {usual:.0f})", 0.5 * usual <= today <= 2 * usual)

### 3. How many of the day's tickets failed a rule?

It takes the share of the newest day's tickets that the Lab 9 quality rules set aside in tickets_quarantine. More than 5% means the data itself is broken. It also alerts when the day has no tickets yet.

In [ ]:
quarantine = table("tickets_quarantine")
parked = quarantine.filter(F.to_date("created_at") == day).count() if quarantine is not None else 0
record("quarantine rate", f"{parked / max(today, 1):.1%} ({parked} of {today})", today > 0 and parked / today <= 0.05)
# Before the quality check (the quality_tickets dataflow) has run, this reads 0%: check 1 is what tells you the day was not handled yet.

### 4. Did the mix of tickets shift?

It works out each AI category's share of the newest day's labelled tickets and of the 30 days before. If any category moves more than 10 percentage points, or the day has no labels yet, it alerts. The table shows every category's move, biggest first.

In [ ]:
enriched = table("tickets_enriched")
def shares(frame):
    """Each AI category's share of the tickets; tickets with no label yet are left out."""
    frame = frame.filter(F.col("ai_category").isNotNull())
    total = frame.count()
    if total == 0:
        return {}
    return {r["ai_category"]: r["count"] / total for r in frame.groupBy("ai_category").count().collect()}
if enriched is not None:
    enriched = enriched.withColumn("d", F.to_date("created_at"))
    now = shares(enriched.filter(F.col("d") == day))
    before = shares(enriched.filter((F.col("d") < day) & (F.col("d") >= F.date_sub(F.lit(day), 30))))
else:
    now, before = {}, {}
if not now or not before:                          # the day has no labels yet, or nothing to compare with
    record("ticket mix", "no labelled tickets for this day yet" if not now else "no earlier labelled days", False)
else:
    moves = {c: round(100 * (now.get(c, 0) - before.get(c, 0)), 1) for c in set(now) | set(before)}
    biggest = max(moves, key=lambda c: abs(moves[c]))
    record("ticket mix", f"{biggest} {moves[biggest]:+} points ({100 * before.get(biggest, 0):.0f}% to {100 * now.get(biggest, 0):.0f}%)",
           abs(moves[biggest]) <= 10)
    display(spark.createDataFrame([(c, round(100 * before.get(c, 0), 1), round(100 * now.get(c, 0), 1), m) for c, m in moves.items()],
                                  "ai_category string, share_before float, share_today float, points float").orderBy(F.desc(F.abs("points"))))

### 5. Did the labelling cost jump?

It reads enrich_log, one row per labelling run, and compares the newest run's cost with the median of the earlier paid runs. More than 5 times that alerts. A normal day often costs $0, because every text already has a stored label, so this check matters when someone changes the prompt or the model.

In [ ]:
log = table("enrich_log")
costs = [r["cost_usd"] for r in log.orderBy("run_at").collect()] if log is not None else []
if not costs:
    record("labelling cost", "no labelling run logged yet", False)
else:
    earlier = [c for c in costs[:-1] if c > 0] or [costs[-1]]
    usual_cost = statistics.median(earlier)
    record("labelling cost", f"${costs[-1]:.6f} (usual ${usual_cost:.6f})", costs[-1] <= 5 * max(usual_cost, 1e-9))

### The report, saved so a person (or an alert rule) can read it

It adds the five results to the health_checks table with the time of the run, so there is a history, and shows them as a table. Then it counts the alerts.

In [ ]:
run_at = time.strftime("%Y-%m-%d %H:%M:%S")
report = spark.createDataFrame([{**r, "run_at": run_at, "day": str(day)} for r in results])
report.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("health_checks")
display(report.select("status", "check", "value", "limit", "owner", "first_action"))
alerts = [r for r in results if r["status"] == "ALERT"]
print(f"{len(alerts)} alert(s) for {day}:", json.dumps([a["check"] for a in alerts]))

### Turn the pipeline red when anything alerts

If any check alerted, this cell stops with an error on purpose. In the pipeline that error turns the run red, and the alert rule sends the message. The report was saved in the cell before, so nothing is lost.

In [ ]:
# The report is saved first. Then a failed step makes the pipeline run show red, so nobody has to open the
# notebook to learn that something is wrong.
if alerts:
    raise RuntimeError(f"{len(alerts)} health alert(s) for {day}: " + "; ".join(a["check"] for a in alerts))